### 🔥 Deep Learning Cave — Chapter 9: The Interview Forge

> *"They don't want to know what you know. They want to know how you think."*

You've built transformers, LLaMA, ViT, JEPA, MoE, distillation pipelines, and Gemma 4 from scratch.

Now it's time to **defend it all** — in a room with someone who helped write the paper.

**Chapter 9** is your sparring partner. Real questions from **OpenAI, Meta FAIR / Superintelligence Lab, Google DeepMind, and Anthropic** — answered with code, math, and the kind of precision that separates researchers from practitioners.

---

#### 🗺️ Your Position in the Cave

```
[■■■■■■■■■■] Chapters 1–8: Built everything from scratch ✓
[■■□□□□□□□□] Chapter 9: Interview Forge ← You are here
 ├── Section A: Transformers & Attention
 ├── Section B: Training, Optimization & Stability
 ├── Section C: RLHF & Alignment
 ├── Section D: Scaling Laws & Architecture Choices
 ├── Section E: Self-Supervised & Representation Learning
 ├── Section F: Evaluation & System Design
 └── Section G: Live Coding Rounds
```

**Labs covered:** OpenAI · Meta FAIR / MSL · Google DeepMind · Anthropic

---

#### How to Use This Notebook

Each question block:
1. **🏢 Source** — which lab typically asks this
2. **❓ Question** — as asked in the interview
3. **🎯 What they're testing** — the real intent
4. **✅ Answer** — concise, precise, reference-level
5. **💻 Code** — verify it yourself (where applicable)
6. **💡 Trap** — the common wrong answer that gets candidates filtered out

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import matplotlib.pyplot as plt

print("Interview Forge loaded. Let's get to work.")
print(f"PyTorch {torch.__version__} | Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

---

## Section A: Transformers & Attention

> *"If you can't derive attention from scratch on a whiteboard, you're not ready."*

---

### Q1 — Derive scaled dot-product attention from first principles. Why the √dₖ scaling?

**🏢 Source:** OpenAI, Anthropic, DeepMind (almost universal opener)

**🎯 What they're testing:** Do you understand WHY the formula is what it is, not just that it exists?

**✅ Answer:**

Given queries Q, keys K, values V (each token projected to dₖ dimensions):

```
Attention(Q, K, V) = softmax( Q Kᵀ / √dₖ ) V
```

**Why √dₖ?**  
Each element of QKᵀ is a dot product of two dₖ-dimensional vectors. If each component has variance 1, the dot product has variance dₖ (sum of dₖ independent terms each with variance 1). So the dot product has std √dₖ.

Without scaling, for large dₖ, these dot products become very large → softmax saturates (outputs near one-hot) → **gradients vanish** (derivative of softmax near 0 and 1 approaches 0).

Dividing by √dₖ normalizes variance back to 1 regardless of dₖ.

**💡 Trap:** Saying "it prevents exploding values" is incomplete. The real reason is **gradient flow through softmax** — it's a training stability argument, not just numerical range.


In [ ]:
# Demonstrate the gradient vanishing problem without scaling
torch.manual_seed(42)

d_k_values = [4, 16, 64, 256, 1024]

print("Effect of d_k on dot product magnitude (std of Q·Kᵀ):")
print(f"{'d_k':>6} | {'No scale (std)':>15} | {'Scaled (std)':>12} | {'Softmax entropy':>15}")
print("-" * 60)

for d_k in d_k_values:
    Q = torch.randn(32, d_k)  # 32 queries
    K = torch.randn(32, d_k)  # 32 keys

    raw_scores   = (Q @ K.T)          # no scaling
    scaled_scores = (Q @ K.T) / math.sqrt(d_k)  # with scaling

    # Entropy of softmax distribution (high entropy = spread out, low = peaked)
    probs = F.softmax(raw_scores, dim=-1)
    entropy = -(probs * (probs + 1e-9).log()).sum(dim=-1).mean().item()

    print(f"{d_k:>6} | {raw_scores.std().item():>15.3f} | {scaled_scores.std().item():>12.3f} | {entropy:>15.3f}")

print()
print("At d_k=1024: raw std ≈ 32 → softmax nearly one-hot → gradient ≈ 0")
print("After scaling: std ≈ 1 regardless of d_k → healthy gradients")

---

### Q2 — What is the difference between encoder-only, decoder-only, and encoder-decoder transformers? When would you use each?

**🏢 Source:** Meta FAIR, Google DeepMind

**🎯 What they're testing:** Architectural understanding + ability to match architecture to task.

**✅ Answer:**

| Architecture | Attention Mask | Pre-training | Best For |
|-------------|---------------|-------------|----------|
| **Encoder-only** (BERT) | Full bidirectional | MLM / masked token | Classification, NER, embeddings |
| **Decoder-only** (GPT, LLaMA) | Causal (lower-triangle) | Next-token prediction | Generation, reasoning, chat |
| **Encoder-Decoder** (T5, BART) | Enc: bidirectional, Dec: causal + cross-attn | Span corruption / denoising | Translation, summarization, seq2seq |

**Key insight:** Decoder-only has **dominated** since GPT-3. Why?
1. Simpler architecture (one unified component)
2. Next-token prediction scales cleanly
3. At inference: same model does classification (few-shot) AND generation
4. Cross-attention in enc-dec doubles KV cache cost

**💡 Trap:** Saying BERT is "better for understanding" — modern decoder-only models with prompting match or beat BERT on classification tasks at scale.

---

### Q3 — Explain Multi-Head Attention. Why multiple heads? What does each head learn?

**🏢 Source:** OpenAI, Anthropic

**🎯 What they're testing:** Intuition beyond the formula.

**✅ Answer:**

MHA runs `h` attention operations **in parallel**, each in a lower-dimensional subspace (dₖ = d_model / h):

```
head_i = Attention(Q Wᵢᴬ, K Wᵢᴷ, V Wᵢᵛ)
MHA    = Concat(head_1, ..., head_h) Wᴼ
```

**Why multiple heads?** A single attention pattern can only express one "relation type" at a time. Multiple heads let the model simultaneously attend to:
- Syntactic dependencies (subject-verb agreement)
- Co-reference ("it" → "the cat")
- Positional offsets (token i attending to i-1)
- Semantic similarity

**Empirical evidence (Voita et al., 2019):** Most heads can be pruned with little quality loss — only a few heads per layer are truly active and specialized.

**💡 Trap:** "More heads = better" — it's a budget split. h heads each get d_model/h dimensions. At some point, heads are too narrow to be useful. Total compute is the same as single-head (same total Q,K,V parameters).

In [ ]:
# Implement MHA from scratch — what the interviewer expects you to write live

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.wq = nn.Linear(d_model, d_model, bias=False)
        self.wk = nn.Linear(d_model, d_model, bias=False)
        self.wv = nn.Linear(d_model, d_model, bias=False)
        self.wo = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, S, D = x.shape
        # Project and split into heads
        Q = self.wq(x).view(B, S, self.n_heads, self.d_k).transpose(1, 2)  # (B,H,S,dk)
        K = self.wk(x).view(B, S, self.n_heads, self.d_k).transpose(1, 2)
        V = self.wv(x).view(B, S, self.n_heads, self.d_k).transpose(1, 2)
        # Attention per head
        out, attn = scaled_dot_product_attention(Q, K, V, mask)
        # Merge heads
        out = out.transpose(1, 2).reshape(B, S, D)
        return self.wo(out), attn


# Test
mha = MultiHeadAttention(d_model=256, n_heads=8)
x = torch.randn(2, 16, 256)
out, attn = mha(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")
print(f"Attention weights: {attn.shape}  (B, H, S, S)")
print(f"\nParams: {sum(p.numel() for p in mha.parameters()):,}")
print(f"  = 4 × d_model² = 4 × {256}² = {4 * 256**2:,}  (all heads combined = 1 big matrix)")

---

### Q4 — What is Grouped Query Attention (GQA)? How does it differ from MHA and MQA?

**🏢 Source:** Meta FAIR (they invented GQA), OpenAI

**🎯 What they're testing:** Knowledge of modern LLM efficiency innovations.

**✅ Answer:**

```
MHA:  H query heads, H key heads, H value heads   → best quality, largest KV cache
MQA:  H query heads, 1 key head,  1 value head    → fastest, quality drop
GQA:  H query heads, G key heads, G value heads   → best tradeoff (G = H/group_size)
```

**KV Cache is the bottleneck at inference.** It grows as: `2 × n_layers × n_kv_heads × head_dim × seq_len × dtype_bytes`

GQA (Ainslie et al., 2023): Group Q heads into G groups; each group shares one K and V head. LLaMA-3 uses GQA with 8 KV heads for 70 heads — 8.75× KV cache reduction.

**Quality:** GQA ≈ MHA (empirically), MQA has noticeable quality drop at scale.

**💡 Trap:** Confusing KV cache size with model parameter size — they're different. GQA saves memory at **inference** (KV cache), not at training (parameter count stays similar).

In [ ]:
# KV cache memory comparison at different sequence lengths
configs = {
    'MHA  (LLaMA-2 70B)':  {'n_kv': 64, 'head_dim': 128, 'n_layers': 80},
    'GQA  (LLaMA-3 70B)':  {'n_kv':  8, 'head_dim': 128, 'n_layers': 80},
    'MQA  (Falcon-7B)':    {'n_kv':  1, 'head_dim': 64,  'n_layers': 32},
}

seq_lens = [1024, 4096, 16384, 65536, 131072]

print(f"{'Model':<25} | ", end='')
print(' | '.join(f'{s//1024}K' for s in seq_lens))
print("-" * 90)

for name, cfg in configs.items():
    sizes = []
    for s in seq_lens:
        # 2 (K+V) × n_layers × n_kv_heads × head_dim × seq_len × 2 bytes (fp16)
        mb = 2 * cfg['n_layers'] * cfg['n_kv'] * cfg['head_dim'] * s * 2 / 1e9
        sizes.append(f"{mb:.1f}GB")
    print(f"{name:<25} | {' | '.join(f'{s:>6}' for s in sizes)}")

print()
print("LLaMA-3 70B with GQA: 8× smaller KV cache vs LLaMA-2 70B (MHA)")
print("This is why GQA is the default in every modern LLM (LLaMA-3, Gemma, Mistral, etc.)")

---

### Q5 — Explain Rotary Position Embeddings (RoPE). Why did it replace sinusoidal PE?

**🏢 Source:** Meta FAIR, OpenAI, DeepMind

**🎯 What they're testing:** Can you explain position encoding without just naming it?

**✅ Answer:**

**Sinusoidal PE (original Transformer):** Add position-dependent vectors to embeddings *before* attention. Problem: positions are added to the token, not to Q/K directly — the position signal gets mixed with content through linear projections.

**RoPE (Su et al., 2021):** Rotate Q and K vectors by an angle proportional to their absolute position. The rotation is applied in 2D subspaces of the head dimension.

**Key property:** The dot product `q_m · k_n` depends only on the **relative position** `(m - n)` — not on absolute positions. This is exactly what attention needs.

```
rotate(q, m) · rotate(k, n) = f(q, k, m-n)   ← relative position!
```

**Why better:**
1. Relative position is baked into attention scores directly
2. Decays gracefully with distance (more distant tokens → weaker attention bias)
3. Long-context extension (YaRN, rope scaling) works cleanly
4. No extra parameters (learned PE needs embedding table)

**💡 Trap:** Saying "RoPE doesn't add a vector" — it *does* modify the representation, just multiplicatively (rotation) rather than additively.

In [ ]:
# Demonstrate the key RoPE property: q_m · k_n depends only on (m-n)

def rope_rotate(x, pos, theta=10000.0):
    """Rotate x at position pos using RoPE."""
    d = x.shape[-1]
    i = torch.arange(0, d, 2).float()
    freqs = 1.0 / (theta ** (i / d))
    angle = pos * freqs  # (d/2,)
    # Build rotation: [cos, -sin; sin, cos] applied per pair
    x_even = x[..., 0::2]
    x_odd  = x[..., 1::2]
    cos_a  = torch.cos(angle)
    sin_a  = torch.sin(angle)
    out_even = x_even * cos_a - x_odd * sin_a
    out_odd  = x_even * sin_a + x_odd * cos_a
    out = torch.zeros_like(x)
    out[..., 0::2] = out_even
    out[..., 1::2] = out_odd
    return out


torch.manual_seed(0)
d = 16
q = torch.randn(d)
k = torch.randn(d)

# Key test: q at pos m, k at pos n → score should depend only on (m-n)
print("Verifying RoPE relative position property:")
print("q_m · k_n should equal q_{m+Δ} · k_{n+Δ} for any Δ\n")

pairs = [(0, 5), (3, 8), (10, 15), (100, 105)]  # all have Δ=5
scores = []
for m, n in pairs:
    q_rot = rope_rotate(q, m)
    k_rot = rope_rotate(k, n)
    score = (q_rot @ k_rot).item()
    scores.append(score)
    print(f"  pos ({m:3d}, {n:3d}), Δ={n-m}: score = {score:.6f}")

print(f"\nAll scores nearly equal: {max(scores) - min(scores):.2e} variation")
print("→ Score depends only on relative distance (n-m=5), not absolute positions!")

---

## Section B: Training, Optimization & Stability

> *"The difference between a paper result and a production model is usually 40 engineering tricks you didn't read about."*

---

### Q6 — Why does modern LLM training use Pre-Norm instead of Post-Norm? What's the difference?

**🏢 Source:** OpenAI, Anthropic, DeepMind

**🎯 What they're testing:** Deep understanding of training stability.

**✅ Answer:**

```python
# Post-Norm (original Transformer, Vaswani 2017)
x = LayerNorm(x + Attention(x))   # normalize AFTER residual

# Pre-Norm (LLaMA, GPT-3, most modern LLMs)
x = x + Attention(LayerNorm(x))   # normalize BEFORE attention
```

**Why Pre-Norm is better for deep networks:**

1. **Gradient flow:** In Post-Norm, the residual path goes through LayerNorm — this can scale down gradients. In Pre-Norm, the residual stream is **unnormalized**, so gradients flow freely from output to input.

2. **Training stability at depth:** Post-Norm networks need careful LR warmup (often failing without it). Pre-Norm trains stably even without warmup.

3. **Layer outputs:** In Post-Norm, the output of each layer has controlled statistics (norm ≈ 1). In Pre-Norm, the residual stream can grow — mitigated by careful init (small weight init on output projections).

**💡 Trap:** Saying Pre-Norm is always better at quality — Post-Norm sometimes achieves slightly better final accuracy at the same depth, but at the cost of training instability. Most practitioners choose stability.

---

### Q7 — What is RMSNorm and why do modern LLMs prefer it over LayerNorm?

**🏢 Source:** Meta FAIR, OpenAI

**🎯 What they're testing:** Practical architecture knowledge.

**✅ Answer:**

**LayerNorm:** Normalizes by subtracting mean and dividing by std, then applies learnable scale (γ) and bias (β).

**RMSNorm (Zhang & Sennrich, 2019):** Drops the mean subtraction step entirely. Just scales by RMS:

```
RMSNorm(x) = x / RMS(x) × γ

where RMS(x) = sqrt(mean(x²) + ε)
```

**Why prefer RMSNorm:**
1. **~10-15% faster** (no mean computation, no bias parameter)
2. **Equal quality** empirically — the re-centering in LayerNorm appears unnecessary
3. **Fewer parameters** (no β bias term)
4. Used in: LLaMA, Gemma, Mistral, Falcon, PaLM

**💡 Trap:** Saying the bias removal is just regularization — it's a speed optimization. The mean-centering step was found empirically not to contribute to quality.

In [ ]:
# Side-by-side: LayerNorm vs RMSNorm

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return (x / rms) * self.weight


d = 512
x = torch.randn(32, 128, d) * 5 + 2  # non-zero mean, large variance

ln  = nn.LayerNorm(d)
rms = RMSNorm(d)

out_ln  = ln(x)
out_rms = rms(x)

print("Input stats:       mean={:.3f}, std={:.3f}".format(x.mean().item(), x.std().item()))
print("LayerNorm output:  mean={:.6f}, std={:.3f}".format(out_ln.mean().item(), out_ln.std().item()))
print("RMSNorm output:    mean={:.3f}, std={:.3f}".format(out_rms.mean().item(), out_rms.std().item()))
print()

ln_params  = sum(p.numel() for p in ln.parameters())
rms_params = sum(p.numel() for p in rms.parameters())
print(f"LayerNorm params: {ln_params}  (γ + β)")
print(f"RMSNorm params:   {rms_params}  (γ only — no bias)")
print(f"Parameter savings: {(ln_params - rms_params)/ln_params*100:.0f}%")
print()
print("RMSNorm does NOT zero-center — but that's fine for transformers")
print("The next linear projection handles any offset; the scale is what matters.")

---

### Q8 — What is SwiGLU and why does it outperform GELU/ReLU in transformers?

**🏢 Source:** Google DeepMind, Meta FAIR

**🎯 What they're testing:** Modern FFN design knowledge.

**✅ Answer:**

Standard FFN: `FFN(x) = max(0, xW₁ + b₁) W₂ + b₂`  (ReLU)

SwiGLU (Noam Shazeer, 2020):
```
SwiGLU(x) = (x W₁ ⊙ SiLU(x W₂)) W₃

SiLU(x) = x · σ(x)   (Swish activation)
```

**Why it's better:**
1. **Gating mechanism:** `W₁ x` acts as a value, `SiLU(W₂ x)` acts as a learned gate. The gate controls which features pass through — adaptive, input-dependent filtering.
2. **Smooth gradient:** SiLU is smooth everywhere (no dead neurons like ReLU)
3. **Empirically superior:** PaLM, LLaMA, Gemma all use SwiGLU and report 1-3% better perplexity vs GELU

**Note on parameter count:** SwiGLU needs 3 weight matrices (W1, W2, W3) vs 2 for standard FFN. To keep the same parameter count, reduce d_ff: typically use `d_ff = 2/3 × 4 × d_model` instead of `4 × d_model`.

**💡 Trap:** Forgetting that SwiGLU uses 3 matrices. If asked to implement it, you'll get points deducted for using 2.

In [ ]:
# Implement and compare FFN variants

class FFN_ReLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


class FFN_SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        # Use 2/3 of d_ff to match parameter count of standard FFN
        d_ff_swi = int(d_ff * 2 / 3)
        self.w1 = nn.Linear(d_model, d_ff_swi, bias=False)  # gate
        self.w2 = nn.Linear(d_model, d_ff_swi, bias=False)  # value
        self.w3 = nn.Linear(d_ff_swi, d_model, bias=False)  # output
    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))  # SwiGLU!


d_model, d_ff = 512, 2048
ffn_relu  = FFN_ReLU(d_model, d_ff)
ffn_swi   = FFN_SwiGLU(d_model, d_ff)

x = torch.randn(4, 32, d_model)
out_r = ffn_relu(x)
out_s = ffn_swi(x)

print(f"ReLU FFN output:   {out_r.shape}  | params: {sum(p.numel() for p in ffn_relu.parameters()):,}")
print(f"SwiGLU FFN output: {out_s.shape}  | params: {sum(p.numel() for p in ffn_swi.parameters()):,}")
print()

# Visualize activation functions
x_plot = torch.linspace(-4, 4, 200)
plt.figure(figsize=(10, 4))
plt.plot(x_plot, F.relu(x_plot),  label='ReLU',  linewidth=2)
plt.plot(x_plot, F.gelu(x_plot),  label='GELU',  linewidth=2)
plt.plot(x_plot, F.silu(x_plot),  label='SiLU / Swish', linewidth=2.5)
plt.title('Activation Functions: ReLU vs GELU vs SiLU (used in SwiGLU)', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()
print("SiLU = x·σ(x): smooth, non-monotonic (slightly negative for x<0), no dead neurons")

---

## Section C: RLHF & Alignment

> *"Scaling got us capable models. Alignment is what makes them useful — and safe."*

---

### Q9 — Explain the full RLHF pipeline. What are its failure modes?

**🏢 Source:** OpenAI (they invented it), Anthropic, Meta

**🎯 What they're testing:** Can you design and critique alignment pipelines?

**✅ Answer:**

```
STEP 1: Supervised Fine-Tuning (SFT)
   Pretrained LLM → fine-tune on human-written demonstrations
   Output: SFT model that imitates human behavior

STEP 2: Reward Model (RM) Training
   - Sample pairs of outputs (y_w, y_l) for same prompt
   - Human annotators rank: y_w ≻ y_l (preferred vs rejected)
   - Train RM: r(x, y_w) > r(x, y_l) via Bradley-Terry model
   - Loss: -log σ(r(x, y_w) - r(x, y_l))

STEP 3: PPO Fine-Tuning
   - Use RM as reward function
   - Policy (LLM) maximizes E[r(x, y)] − β·KL(π || π_ref)
   - KL penalty prevents policy from drifting too far from SFT model
```

**Known failure modes:**
1. **Reward hacking:** Policy finds outputs that maximize RM score but aren't actually good (verbose, sycophantic)
2. **Distributional shift:** RM trained on SFT outputs; PPO policy generates different text → RM unreliable out-of-distribution
3. **Sycophancy:** Model learns to agree with user's stated beliefs rather than be truthful
4. **Over-refusal:** Model becomes too cautious, refusing benign requests
5. **Preference annotation noise:** Human preferences are inconsistent, ambiguous, culturally biased

**DPO (Direct Preference Optimization):** Eliminates the separate RM. Directly trains on preference pairs using a reparameterized objective. Simpler, more stable, increasingly preferred (used by Zephyr, Gemma-IT).

**💡 Trap:** Treating RLHF as solved — the field has moved to DPO, GRPO, and process-based reward models (PRMs). Know these alternatives.

---

### Q10 — What is DPO and how does it differ from PPO-based RLHF?

**🏢 Source:** Anthropic, Meta, OpenAI

**🎯 What they're testing:** Knowledge of current alignment methods.

**✅ Answer:**

**DPO key insight:** The optimal RLHF policy has a closed-form expression:

```
π*(y|x) = π_ref(y|x) · exp(r*(x,y)/β) / Z(x)
```

This lets you express the reward in terms of policy log-ratios — **eliminating the need for an explicit reward model**:

```
DPO Loss = -E[ log σ( β · log(π_θ(y_w|x)/π_ref(y_w|x))
                    - β · log(π_θ(y_l|x)/π_ref(y_l|x)) ) ]
```

| Aspect | PPO-RLHF | DPO |
|--------|----------|-----|
| Reward model | Required (separate training) | Not needed |
| Training stability | PPO is notoriously unstable | Much more stable |
| Compute | 4 models in memory (policy, ref, RM, value) | 2 models (policy, ref) |
| Quality | Marginally better at scale | Slightly worse at very large scale |
| Use today | o1/o3 reasoning training | Most open-source IT models |

**💡 Trap:** Saying DPO is strictly better — OpenAI's o1/o3 models still use PPO with process-based reward models (PRMs) for reasoning tasks where DPO struggles.

---

## Section D: Scaling Laws & Architecture Choices

> *"The Chinchilla paper invalidated GPT-3 the day it was published. Know it cold."*

---

### Q11 — What are the Chinchilla scaling laws? What did they change about how we train LLMs?

**🏢 Source:** Google DeepMind (they wrote the paper), OpenAI, Meta

**🎯 What they're testing:** Foundational understanding of how training budget decisions are made.

**✅ Answer:**

**Hoffmann et al., 2022 (DeepMind):** Given a compute budget C (FLOPs), the optimal model size N and token count D satisfy:

```
N_opt ∝ C^0.5
D_opt ∝ C^0.5

→ N_opt ≈ 20 × D_opt  (tokens ≈ 20× parameters)
```

**The GPT-3 lesson:** GPT-3 (175B params) was trained on only ~300B tokens — vastly undertrained according to Chinchilla. A 12B model trained on 2.4T tokens would have matched GPT-3 performance at the same compute.

**What changed:**
1. Models got smaller + trained longer (LLaMA: 7B on 1T tokens, LLaMA-2: 7B on 2T)
2. "Small but well-trained" models became viable for deployment
3. Compute-optimal ≠ inference-optimal: Gemma 2 trains 50× past compute-optimal because inference cost dominates

**💡 Trap:** Applying Chinchilla laws blindly. They assume you train once and deploy once. If you serve 1M users, inference cost dominates — then train a smaller model for much longer (Llama philosophy).

In [ ]:
# Visualize Chinchilla scaling: optimal N and D for a given compute budget

# Approximate: FLOPs ≈ 6 × N × D (for transformer training)
# Chinchilla optimal: N = D → C = 6 N²  →  N = sqrt(C/6)

compute_budgets_flops = np.logspace(18, 25, 50)  # 10^18 to 10^25 FLOPs

# Chinchilla optimal
N_opt = np.sqrt(compute_budgets_flops / 6)
D_opt = compute_budgets_flops / (6 * N_opt)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].loglog(compute_budgets_flops, N_opt / 1e9, 'b-', linewidth=2.5, label='Optimal model size (Chinchilla)')
axes[0].loglog(compute_budgets_flops, D_opt / 1e9, 'r--', linewidth=2.5, label='Optimal token count')

# Mark known models
models = {
    'GPT-3\n(undertrained)': (3.14e23, 175, 300),
    'Chinchilla': (5.76e23, 70, 1400),
    'LLaMA-2 7B': (1.0e23, 7, 2000),
    'Gemma 2B': (1.5e23, 2.3, 3000),
}

for name, (C, N, D) in models.items():
    axes[0].plot(C, N, 'o', markersize=10)
    axes[0].annotate(name, (C, N), textcoords='offset points', xytext=(5, 5), fontsize=8)

axes[0].set_xlabel('Compute Budget (FLOPs)')
axes[0].set_ylabel('Billions')
axes[0].set_title('Chinchilla Scaling Laws:\nOptimal N and D per compute budget', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Right: token:param ratio for known models
model_names = ['GPT-3', 'Chinchilla', 'LLaMA-1\n7B', 'LLaMA-2\n70B', 'Gemma 2B\n(Distilled)']
ratios = [1.7, 20, 143, 28, 1300]
colors = ['#e74c3c', '#27ae60', '#3498db', '#3498db', '#9b59b6']

bars = axes[1].bar(model_names, ratios, color=colors, alpha=0.8, edgecolor='#2c3e50')
axes[1].axhline(20, color='green', linestyle='--', linewidth=2, label='Chinchilla optimal (20×)')
axes[1].set_ylabel('Tokens per Parameter')
axes[1].set_title('Tokens:Parameter Ratio\n(Chinchilla optimal = 20×)', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')
for bar, ratio in zip(bars, ratios):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{ratio}×', ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

print("GPT-3: 1.7× tokens/param → massively undertrained")
print("LLaMA-1 7B: 143× → over-trained for compute, optimal for inference")
print("Gemma 2B: 1300× → 50× past compute optimal, enabled by distillation")

---

### Q12 — What is Flash Attention? Why does vanilla attention not scale?

**🏢 Source:** OpenAI, Meta, DeepMind

**🎯 What they're testing:** Systems understanding of attention bottlenecks.

**✅ Answer:**

**Vanilla attention bottleneck:** The attention matrix S = QKᵀ is (N × N). For N=128K tokens:
- Memory: 128K × 128K × 2 bytes (fp16) = **32GB** per layer per head
- This is a **memory bandwidth** problem, not a compute problem

**Flash Attention (Dao et al., 2022):** Reorders the attention computation to avoid materializing the full N×N matrix.

**Key insight — tiling:** Split Q, K, V into blocks that fit in SRAM (fast, on-chip memory). Process block by block, computing the softmax incrementally using the "online softmax" trick.

```
Standard: HBM read Q,K,V → compute S=QKᵀ → write S → read S → softmax → write → read → output
           IO: O(N²)   — N² matrix must be written/read from slow HBM

FlashAttn: Tile into blocks → compute+softmax in SRAM → only write final output
           IO: O(N)    — never materialize N×N in slow memory
```

**Results:** 3-20× faster, O(N) memory instead of O(N²), exact same output (not approximate).

**Flash Attention 2 & 3:** Further optimizations for parallelism across heads and sequence length.

**💡 Trap:** Thinking Flash Attention reduces compute FLOPs — it doesn't. It reduces memory **IO**. The FLOPs are the same; the speedup comes from not hitting slow HBM.

---

## Section E: Self-Supervised & Representation Learning

---

### Q13 — Explain the difference between contrastive, generative, and predictive SSL. Which does JEPA use?

**🏢 Source:** Meta FAIR (LeCun's team), DeepMind

**🎯 What they're testing:** Understanding the landscape of self-supervised learning — Meta FAIR cares deeply about this.

**✅ Answer:**

| Paradigm | Method | What it learns | Limitation |
|----------|--------|----------------|------------|
| **Contrastive** | SimCLR, MoCo, CLIP | Attract augmented views, repel random | Needs negatives, sensitive to augmentations |
| **Generative** | MAE, autoencoders | Reconstruct masked/noisy input | Wastes capacity on pixel-level details |
| **Predictive (JEPA)** | I-JEPA, V-JEPA | Predict abstract representations of masked regions | No reconstruction, no negatives |

**JEPA's key insight (LeCun's "A Path Towards Autonomous Machine Intelligence"):**
- Predict in **latent space**, not pixel space
- Force the model to learn **semantic understanding**, not texture reconstruction
- EMA target encoder provides stable targets without collapse

**Why JEPA > MAE (LeCun's argument):**
MAE spends 80% of its capacity learning to predict low-level pixels that contain no semantic information. JEPA predicts high-level features — the model must understand the scene to predict what's there.

**💡 Trap:** Saying JEPA avoids collapse because of the EMA alone — it's the combination of EMA + large semantic block masking + predicting in representation space. Any one of these alone is insufficient.

---

### Q14 — What is representation collapse in SSL and how is it prevented?

**🏢 Source:** Meta FAIR, DeepMind

**🎯 What they're testing:** Deep SSL understanding.

**✅ Answer:**

**Collapse:** The trivial solution to SSL is outputting the same constant vector for every input — loss = 0, zero information learned.

**Prevention strategies:**

| Method | Mechanism | How it prevents collapse |
|--------|-----------|-------------------------|
| SimCLR | Negative samples | Explicitly push apart different samples |
| BYOL | Asymmetric architecture (predictor head) | Breaks symmetry — encoder and predictor can't co-collapse |
| MoCo | Momentum encoder (EMA) | Target changes slowly — online encoder must move to match |
| VICReg | Variance + covariance regularization | Directly penalize collapsed/correlated representations |
| JEPA | EMA target + semantic block masking | Stable targets + hard prediction task forces diversity |

**Mathematical view (VICReg):** Penalize if the covariance matrix of the batch representations is not identity — forces spread across all dimensions.

**💡 Trap:** Thinking EMA alone prevents collapse — BYOL without the predictor head collapses. The predictor creates the necessary asymmetry.

---

## Section F: Evaluation & System Design

---

### Q15 — Design an LLM evaluation pipeline for a coding assistant. What metrics would you track?

**🏢 Source:** OpenAI (they literally build Copilot), Anthropic, Meta

**🎯 What they're testing:** ML systems design — can you design end-to-end evaluation?

**✅ Answer:**

**Levels of evaluation (most interviewers want all four):**

```
Level 1: Offline Benchmarks (fast, automated)
   - HumanEval / MBPP: function-level code generation (pass@1, pass@k)
   - SWE-bench: real GitHub issue resolution
   - CRUXEval: code reasoning (input→output, output→input)
   - LiveCodeBench: contamination-free, rolling benchmark

Level 2: Execution-Based (trust the tests, not the judge)
   - Sandboxed execution (Docker/e2b) of generated code
   - Run the actual test suite — binary pass/fail
   - Measure: syntax errors, runtime errors, test failures separately

Level 3: LLM-as-Judge (for qualitative aspects)
   - Readability, documentation quality, style consistency
   - Use GPT-4 or Claude as judge (G-Eval framework)
   - Risk: judge sycophancy — mitigate with pairwise comparison

Level 4: Human Evaluation + A/B Testing (ground truth)
   - Shadow traffic: show output to users, measure acceptance rate
   - User edits: did they accept/edit/delete the suggestion?
   - Task completion rate: did the feature get shipped?
```

**Key metrics to report:**
- `pass@1`, `pass@10` (with temperature sampling)
- Edit distance between suggestion and final code
- Latency p50/p99 (user experience)
- False refusal rate (how often model declines valid requests)

**💡 Trap:** Only using offline benchmarks. They're contaminated, static, and don't measure what users actually care about. The best eval is production behavior.

---

### Q16 — What is perplexity? What are its limitations as an LLM evaluation metric?

**🏢 Source:** All labs

**🎯 What they're testing:** Statistical understanding + critical thinking about metrics.

**✅ Answer:**

**Perplexity:** Exponential of average cross-entropy loss on a held-out test set:

```
PPL = exp( -1/N · Σᵢ log P(xᵢ | x₁,...,xᵢ₋₁) )

Lower PPL = model assigns higher probability to the text = better
```

**Limitations:**
1. **Domain mismatch:** PPL on Wikipedia vs PPL on code are incomparable (vocabulary difficulty differs)
2. **Tokenizer sensitivity:** Same model, different tokenizer → very different PPL (different denominator N)
3. **Doesn't measure reasoning:** A model with PPL=10 can still fail completely at math
4. **Doesn't measure factuality:** Model can assign high probability to fluent falsehoods
5. **Compression vs understanding:** PPL measures text compression, not world knowledge

**When PPL IS useful:**
- Comparing two versions of the same model on the same tokenizer and test set
- Detecting catastrophic forgetting after fine-tuning
- Quick ablation during pretraining

**💡 Trap:** Using PPL to compare models with different tokenizers (e.g., GPT-4 vs LLaMA-3) — meaningless.

In [ ]:
# Implement perplexity computation — what they ask you to code live

def compute_perplexity(model, token_ids, stride=512):
    """Compute perplexity using sliding window (handles long sequences).

    stride: overlap between windows (avoids edge effects from limited context)
    """
    model.eval()
    device = next(model.parameters()).device
    token_ids = token_ids.to(device)
    seq_len = token_ids.shape[1]
    max_len = 64  # model context (demo)

    nll_sum = 0.0
    n_tokens = 0

    for begin in range(0, seq_len - 1, stride):
        end = min(begin + max_len, seq_len)
        input_ids  = token_ids[:, begin:end - 1]
        target_ids = token_ids[:, begin + 1:end]

        with torch.no_grad():
            logits = model(input_ids)          # (B, S, V)
            loss   = F.cross_entropy(
                logits.reshape(-1, logits.shape[-1]),
                target_ids.reshape(-1),
                reduction='sum'
            )

        nll_sum += loss.item()
        n_tokens += target_ids.numel()

    avg_nll = nll_sum / n_tokens
    ppl = math.exp(avg_nll)
    return ppl, avg_nll


# Demo with tiny random model
class TinyLM(nn.Module):
    def __init__(self, vocab=100, d=64, n=2):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.layers = nn.Sequential(*[nn.Linear(d, d) for _ in range(n)])
        self.head = nn.Linear(d, vocab)
    def forward(self, x):
        return self.head(self.layers(self.embed(x)))

model = TinyLM()
tokens = torch.randint(0, 100, (1, 128))
ppl, nll = compute_perplexity(model, tokens)

print(f"Random model perplexity: {ppl:.2f}")
print(f"Average NLL:             {nll:.4f}")
print(f"Expected PPL (random):   ~{100:.1f}  (vocab size = 100, uniform = 100 PPL)")
print()
print("Rule of thumb perplexity values:")
print("  GPT-2 (1.5B) on WikiText-103:  ~18")
print("  LLaMA-3 70B  on same set:       ~4-6")
print("  Perfect model:                  = 1")
print("  Random (vocab=50K):             = 50000")

---

## Section G: Live Coding Rounds

> *"The coding round isn't about code. It's about how you think when you're under pressure."*

---

### Q17 — Implement top-k and top-p (nucleus) sampling from scratch.

**🏢 Source:** OpenAI, Anthropic, Meta — extremely common coding round question

**🎯 What they're testing:** Can you implement LLM decoding strategies cleanly?

In [ ]:
def top_k_sampling(logits: torch.Tensor, k: int, temperature: float = 1.0) -> torch.Tensor:
    """Sample from top-k tokens only.

    1. Keep only the k highest logits
    2. Set all others to -inf (excluded from softmax)
    3. Sample from the resulting distribution
    """
    logits = logits / temperature

    # Find the k-th largest value as a threshold
    top_k_values, _ = torch.topk(logits, k, dim=-1)
    threshold = top_k_values[..., -1, None]  # k-th largest

    # Zero out everything below threshold
    filtered = logits.masked_fill(logits < threshold, float('-inf'))

    probs = F.softmax(filtered, dim=-1)
    return torch.multinomial(probs, num_samples=1).squeeze(-1)


def top_p_sampling(logits: torch.Tensor, p: float, temperature: float = 1.0) -> torch.Tensor:
    """Nucleus sampling: sample from the smallest set of tokens with cumulative prob >= p.

    1. Sort tokens by probability (descending)
    2. Compute cumulative probability
    3. Keep tokens until cumulative prob >= p
    4. Sample from kept tokens
    """
    logits = logits / temperature
    probs = F.softmax(logits, dim=-1)

    # Sort descending
    sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)

    # Cumulative probabilities
    cumprobs = sorted_probs.cumsum(dim=-1)

    # Remove tokens once cumulative prob exceeds p
    # Shift by 1 to include the token that pushes over p
    remove_mask = (cumprobs - sorted_probs) >= p
    sorted_probs[remove_mask] = 0.0

    # Renormalize
    sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True)

    # Sample and map back to original indices
    sampled_sorted = torch.multinomial(sorted_probs, num_samples=1)
    sampled_idx = sorted_idx.gather(-1, sampled_sorted)

    return sampled_idx.squeeze(-1)


# Test and visualize
torch.manual_seed(42)
vocab_size = 20
logits = torch.randn(1, vocab_size) * 2
probs  = F.softmax(logits, dim=-1).squeeze()

# Show which tokens are kept by each strategy
k = 5
p = 0.9

sorted_probs, sorted_idx = probs.sort(descending=True)
cumprobs = sorted_probs.cumsum(0)
nucleus_size = (cumprobs < p).sum().item() + 1

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Full distribution
axes[0].bar(range(vocab_size), probs.numpy(), color='#3498db', alpha=0.7)
axes[0].set_title('Full Softmax Distribution', fontweight='bold')
axes[0].set_xlabel('Token ID')

# Top-k
topk_mask = probs >= probs.topk(k).values[-1]
colors_k = ['#e74c3c' if m else '#95a5a6' for m in topk_mask.numpy()]
axes[1].bar(range(vocab_size), probs.numpy(), color=colors_k, alpha=0.8)
axes[1].set_title(f'Top-k (k={k}): Red = kept tokens', fontweight='bold')
axes[1].set_xlabel('Token ID')

# Top-p: show sorted cumulative
axes[2].bar(range(len(sorted_probs)), sorted_probs.numpy(), color='#2ecc71', alpha=0.8)
axes[2].plot(range(len(cumprobs)), cumprobs.numpy(), 'r-', linewidth=2, label='Cumulative prob')
axes[2].axhline(p, color='navy', linestyle='--', linewidth=2, label=f'p={p} threshold')
axes[2].axvline(nucleus_size, color='orange', linestyle='--', linewidth=2, label=f'Nucleus size={nucleus_size}')
axes[2].set_title(f'Top-p (p={p}) Sampling — Sorted', fontweight='bold')
axes[2].set_xlabel('Rank (sorted by prob)')
axes[2].legend(fontsize=8)

plt.suptitle('Decoding Strategies: Greedy vs Top-k vs Top-p (Nucleus)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Quick correctness test
k_sample = top_k_sampling(logits, k=5)
p_sample = top_p_sampling(logits, p=0.9)
print(f"Top-k sample: token {k_sample.item()} (prob {probs[k_sample].item():.3f})")
print(f"Top-p sample: token {p_sample.item()} (prob {probs[p_sample].item():.3f})")

---

### Q18 — Implement a simple byte-pair encoding (BPE) tokenizer.

**🏢 Source:** OpenAI (GPT uses BPE), Meta (LLaMA uses SentencePiece BPE)

**🎯 What they're testing:** Understanding of tokenization — often asked to debug slow LLM behavior.

In [ ]:
from collections import Counter, defaultdict

def get_vocab(corpus):
    """Initialize vocab: each word split into characters + </w> end-of-word marker."""
    vocab = Counter()
    for word in corpus.split():
        # ' '.join splits word into characters; </w> marks word boundary
        chars = ' '.join(list(word)) + ' </w>'
        vocab[chars] += 1
    return vocab


def get_pairs(vocab):
    """Count all adjacent symbol pairs across the vocabulary."""
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs


def merge_pair(pair, vocab):
    """Merge the most frequent pair in all words."""
    new_vocab = {}
    bigram = ' '.join(pair)  # e.g. ('l', 'o') → 'l o'
    replacement = ''.join(pair)  # 'lo'
    for word, freq in vocab.items():
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = freq
    return new_vocab


def train_bpe(corpus, num_merges=15):
    vocab = get_vocab(corpus)
    merges = []

    print(f"Initial vocab ({len(vocab)} words):")
    for w, c in list(vocab.items())[:3]:
        print(f"  '{w}': {c}")
    print()

    for i in range(num_merges):
        pairs = get_pairs(vocab)
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        vocab = merge_pair(best, vocab)
        merges.append(best)
        print(f"Merge {i+1:2d}: {best[0]!r} + {best[1]!r} → {''.join(best)!r}  (freq={pairs[best]})")

    return vocab, merges


corpus = "low lower lowest new newer newest"
vocab, merges = train_bpe(corpus, num_merges=10)

print(f"\nFinal vocab:")
for w in vocab:
    print(f"  {w}")

---

### Q19 — Implement beam search for sequence generation.

**🏢 Source:** Google DeepMind, Meta

**🎯 What they're testing:** Understanding of autoregressive decoding tradeoffs.

In [ ]:
def beam_search(logits_fn, prompt, beam_width=3, max_new_tokens=10, eos_token=1, vocab_size=50):
    """Beam search decoding.

    Maintains `beam_width` candidate sequences, always expanding the best.

    Args:
        logits_fn: function(token_ids) → (B, S, V) logits
        prompt: (S,) starting token ids
        beam_width: number of beams to maintain
        max_new_tokens: max tokens to generate
        eos_token: end-of-sequence token id

    Returns:
        list of (score, token_ids) sorted by score descending
    """
    # beams: list of (log_prob, token_sequence)
    beams = [(0.0, prompt.tolist())]
    completed = []

    for step in range(max_new_tokens):
        all_candidates = []

        for log_prob, seq in beams:
            if seq[-1] == eos_token:
                completed.append((log_prob, seq))
                continue

            # Get next-token distribution
            input_ids = torch.tensor(seq).unsqueeze(0)
            with torch.no_grad():
                next_logits = logits_fn(input_ids)[:, -1, :]  # (1, V)

            log_probs = F.log_softmax(next_logits, dim=-1).squeeze(0)  # (V,)

            # Expand: take top beam_width candidates
            top_lp, top_tok = torch.topk(log_probs, beam_width)
            for lp, tok in zip(top_lp.tolist(), top_tok.tolist()):
                all_candidates.append((log_prob + lp, seq + [tok]))

        if not all_candidates:
            break

        # Keep top beam_width candidates
        all_candidates.sort(key=lambda x: x[0], reverse=True)
        beams = all_candidates[:beam_width]

    completed.extend(beams)
    completed.sort(key=lambda x: x[0], reverse=True)
    return completed[:beam_width]


# Demo with tiny random LM
torch.manual_seed(0)
tiny_lm = TinyLM(vocab=50, d=32, n=1)
prompt = torch.tensor([5, 10, 3])  # 3-token prompt

results = beam_search(
    logits_fn=tiny_lm,
    prompt=prompt,
    beam_width=3,
    max_new_tokens=5,
    eos_token=1,
    vocab_size=50
)

print("Beam Search results (top 3 beams):")
for rank, (score, tokens) in enumerate(results):
    generated = tokens[len(prompt):]
    print(f"  Beam {rank+1}: score={score:.3f} | generated tokens: {generated}")

print()
print("Key tradeoffs:")
print("  beam_width=1  → greedy decoding (fastest, lowest quality)")
print("  beam_width=4  → standard NMT setting")
print("  beam_width→∞  → exhaustive search (intractable)")
print("  Beam search ≠ optimal for open-ended generation (repetition, 'generic' outputs)")
print("  → Modern LLMs use top-p/top-k sampling instead")

---

### Q20 — What is the vanishing/exploding gradient problem? How does the transformer architecture address it?

**🏢 Source:** DeepMind, Anthropic, Meta

**🎯 What they're testing:** Foundational training dynamics understanding.

**✅ Answer:**

**Vanishing gradient:** In deep networks, gradients ∂L/∂W involve products of Jacobians across layers. If each layer shrinks the gradient (spectral norm < 1), the gradient → 0 exponentially with depth. Early layers stop learning.

**Exploding gradient:** Opposite — gradient grows exponentially, causing NaN losses.

**How transformers address it:**

| Mechanism | How it helps |
|-----------|-------------|
| **Residual connections** | Gradient has a direct path: ∂L/∂xₗ = ∂L/∂xₙ + path through layers. Even if per-layer gradients vanish, the residual highway carries gradient directly. |
| **LayerNorm / RMSNorm** | Normalizes activations → prevents magnitude explosion through depth |
| **Gradient clipping** | `clip_grad_norm_(params, 1.0)` — hard cap on gradient norm |
| **Small init (output proj)** | Initialize attention and FFN output projections to ≈ 0 → identity-like residuals at init |
| **Adam optimizer** | Adaptive learning rates — different params update at appropriate scales |

**Key formula — residual gradient:**
```
x_out = x_in + F(x_in)
∂x_out/∂x_in = I + ∂F/∂x_in
```
The identity term `I` ensures gradients never vanish through the residual path, even if `∂F/∂x_in ≈ 0`.

**💡 Trap:** Saying LayerNorm prevents vanishing gradients — it normalizes activations, not gradients. The residual connection is the primary mechanism.

---

### Quick-Fire: The 90-Second Answers

These come up constantly and must be answered in under 2 minutes:

| Question | Answer in one sentence |
|----------|------------------------|
| **What is weight tying?** | Sharing the embedding matrix weight with the output projection head — saves ~50% embedding params, often improves performance |
| **What is gradient checkpointing?** | Recompute activations during backward pass instead of storing them — trades compute for memory (enables 5-8× larger batch sizes) |
| **What is mixed precision training?** | Store weights in fp32 but do forward/backward in fp16/bf16 — 2× speedup, 2× memory reduction, same quality if done correctly |
| **What is LoRA?** | Approximate full fine-tuning by decomposing the weight update as ΔW = A·B (low-rank) — only A and B are trained; 99% of params frozen |
| **What is speculative decoding?** | Use a small fast draft model to generate k tokens; large model verifies all k in one forward pass — 2-3× speedup with identical outputs |
| **What is KV cache?** | Store computed K,V tensors during autoregressive generation — avoids recomputing them for every new token (O(n²) → O(n)) |
| **What is temperature in sampling?** | Divide logits by T before softmax: T>1 = flatter (more random), T<1 = peakier (more greedy), T=0 = greedy argmax |
| **Explain bias-variance tradeoff** | High bias = model too simple (underfits). High variance = model too complex (overfits). Sweet spot: low both via right capacity + regularization |
| **What is hallucination in LLMs?** | Model generates fluent text that is factually incorrect or fabricated — stems from autoregressive training rewarding fluency, not factuality |
| **What is mixture-of-experts?** | Route each token to only K of N expert FFN sublayers — enables massive model capacity at constant compute cost per token |


---

### Summary: The Cheat Sheet

```
ATTENTION
  Scaled dot-product:  Q Kᵀ / √dₖ  — scale prevents softmax saturation
  MHA:                 h heads, each dₖ = d_model/h
  GQA:                 h Q heads, g KV heads — saves KV cache, LLaMA-3/Gemma
  Flash Attention:     Tiled, avoids materializing N×N in HBM — IO-optimal

POSITIONAL ENCODING
  Sinusoidal:          Absolute, not learnable, original Transformer
  RoPE:                Relative, rotates Q/K, dominant in modern LLMs
  Dual RoPE:           Gemma 4 — different θ for local vs global attention

NORMALIZATION
  Post-Norm:           Original Transformer, less stable
  Pre-Norm:            Modern default, better gradient flow
  LayerNorm:           Mean + variance normalization, γ and β
  RMSNorm:             Drop mean subtraction, 10-15% faster, same quality

FFN
  ReLU/GELU:           Standard, 2 matrices
  SwiGLU:              Gated, 3 matrices, best quality, LLaMA/Gemma/PaLM

TRAINING
  Chinchilla:          Train 20 tokens/param for compute-optimal
  Scaling laws:        Loss ∝ N^-0.5 and D^-0.5 (model size × data)
  RLHF:                SFT → Reward Model → PPO
  DPO:                 Skips reward model, directly trains on preferences

LABS
  OpenAI:              Scaling, RLHF, o1/o3 test-time compute
  Meta FAIR/MSL:       JEPA, LLaMA, open-source, efficiency
  Google DeepMind:     Chinchilla, AlphaFold, Gemma, Flash Attention
  Anthropic:           Constitutional AI, interpretability, safety
```

---

🔥 **You built it all from scratch in Chapters 1-8. Now you can explain it under pressure in Chapter 9.**

That's the difference between knowing it and owning it. 🗿